In [1]:
import json
import pandas as pd
from datasets import load_dataset

# Домены естественных наук
NATURAL_SCIENCE_DOMAINS = {
    "high_school_biology",
    "high_school_chemistry",
    "high_school_physics",
    "high_school_geography",
    "high_school_environmental_science",
    "college_biology",
    "college_chemistry",
    "college_physics",
    "college_medicine",
    "anatomy",
    "astronomy",
    "conceptual_physics",
    "professional_medicine",
    "medical_genetics",
    "clinical_knowledge",
    "human_aging",
    "nutrition",
    "virology",
    "human_sexuality",
    "electrical_engineering",
}

SAMPLE_SIZE = 500
RANDOM_SEED = 42
OUTPUT_FILE = "natural_science_sample_500.json"


def load_rummlu():
    """Загружает ruMMLU из ai-forever/MERA и разворачивает поля."""
    print("Загружаем ruMMLU из ai-forever/MERA ...")
    ds = load_dataset("ai-forever/MERA", name="rummlu")

    # public_test 10k примеров с ответами, используем для оценки
    df = ds["public_test"].to_pandas()
    print(f"  Загружено примеров: {len(df)}")

    # Разворачиваем inputs: text, option_a/b/c/d, subject
    inputs_expanded = pd.json_normalize(df["inputs"])
    df = pd.concat([df.drop(columns=["inputs"]), inputs_expanded], axis=1)

    # Достаём domain из meta
    df["domain"] = df["meta"].apply(
        lambda m: m.get("domain", "") if isinstance(m, dict) else ""
    )

    domains = sorted(df["domain"].dropna().unique())
    print(f"  Уникальных доменов: {len(domains)}")
    return df, domains


def filter_natural_sciences(df: pd.DataFrame) -> pd.DataFrame:
    """Оставляет только примеры из категорий естественных наук."""
    mask = df["domain"].isin(NATURAL_SCIENCE_DOMAINS)
    filtered = df[mask].reset_index(drop=True)
    print(f"Естественные науки: {len(df)} → {len(filtered)} примеров")
    if len(filtered) == 0:
        print("  ВНИМАНИЕ: 0 примеров! Все домены в датасете:")
        print(sorted(df["domain"].unique()))
    return filtered


def sample_and_save(science_df: pd.DataFrame, n: int = SAMPLE_SIZE,
                    seed: int = RANDOM_SEED, output_file: str = OUTPUT_FILE):
    """
    Выбирает n примеров из science_df с равномерным распределением по доменам
    (стратифицированная выборка: n // num_domains примеров из каждого домена).

    Сохраняет два файла:
      - output_file              : сами 500 примеров (список dict)
      - output_file + '.ids.json': список оригинальных idx для исключения
                                   при оценке на бенчмарке

    Returns:
        sample_df   : DataFrame с выбранными примерами
        excluded_ids: список оригинальных индексов выборки
    """
    assert len(science_df) >= n, (
        f"В датасете только {len(science_df)} примеров, запрошено {n}"
    )

    domains = science_df["domain"].unique()
    per_domain = n // len(domains)
    remainder = n % len(domains)

    print(f"\nСтратифицированная выборка: {len(domains)} доменов × {per_domain} = {per_domain * len(domains)} примеров"
          + (f" + {remainder} дополнительных" if remainder else ""))

    chunks = []
    for i, domain in enumerate(sorted(domains)):
        domain_df = science_df[science_df["domain"] == domain]
        # Первые remainder доменов берут на 1 пример больше, если n не делится ровно
        take = per_domain + (1 if i < remainder else 0)
        if len(domain_df) < take:
            print(f"  ВНИМАНИЕ: домен '{domain}' имеет только {len(domain_df)} примеров, берём все")
            take = len(domain_df)
        chunks.append(domain_df.sample(n=take, random_state=seed))

    sample_df = pd.concat(chunks).sample(frac=1, random_state=seed).reset_index(drop=True)

    # Сохраняем оригинальные индексы для последующего исключения
    if "idx" in sample_df.columns:
        excluded_ids = sample_df["idx"].tolist()
    else:
        # Восстанавливаем индексы из science_df через merge
        excluded_ids = (
            science_df
            .sample(n=n, random_state=seed)
            .index.tolist()
        )

    ids_file = output_file.replace(".json", "") + ".ids.json"
    with open(ids_file, "w", encoding="utf-8") as f:
        json.dump(excluded_ids, f, ensure_ascii=False, indent=2)
    print(f"Индексы для исключения сохранены → {ids_file}  ({len(excluded_ids)} id)")

    # Сохраняем сами примеры
    # meta — dict, оставляем как есть; outputs — int/str
    records = sample_df.to_dict(orient="records")
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"Выборка сохранена → {output_file}  ({len(records)} примеров)")

    return sample_df, excluded_ids


def load_excluded_ids(ids_file: str = OUTPUT_FILE.replace(".json", "") + ".ids.json"):
    """Загружает список idx, которые нужно убрать из бенчмарка."""
    with open(ids_file, "r", encoding="utf-8") as f:
        return set(json.load(f))


def filter_out_sampled(df: pd.DataFrame, ids_file: str = None) -> pd.DataFrame:
    """
    Убирает из df строки, чьи idx попали в обучающую выборку.
    Используй в коде оценки, чтобы не тестироваться на уже виденных данных.

    Пример:
        eval_df = filter_out_sampled(science_df)
    """
    if ids_file is None:
        ids_file = OUTPUT_FILE.replace(".json", "") + ".ids.json"

    excluded = load_excluded_ids(ids_file)
    id_col = "idx" if "idx" in df.columns else df.index.name or df.index

    if "idx" in df.columns:
        before = len(df)
        df_filtered = df[~df["idx"].isin(excluded)].reset_index(drop=True)
    else:
        before = len(df)
        df_filtered = df[~df.index.isin(excluded)].reset_index(drop=True)

    print(f"filter_out_sampled: {before} → {len(df_filtered)} примеров "
          f"(убрано {before - len(df_filtered)})")
    return df_filtered

# Основной запуск
if __name__ == "__main__":
    # Загружаем
    public_df, all_domains = load_rummlu()

    # Фильтрация по наукам
    science_df = filter_natural_sciences(public_df)

    # Статистика по доменам естественных наук
    print("\nПримеров по доменам естественных наук:")
    sci_counts = science_df["domain"].value_counts()
    print(sci_counts.to_string())
    print(f"\nИтого по наукам: {len(science_df)} примеров")

    # Сэмплируем 500 случайных примеров и сохраняем
    sample_df, excluded_ids = sample_and_save(science_df)

    print(f"\n Примеры по доменам в выборке:")
    print(sample_df["domain"].value_counts().to_string())

Загружаем ruMMLU из ai-forever/MERA ...


README.md:   0%|          | 0.00/137k [00:00<?, ?B/s]

data/rummlu/train.jsonl:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/845k [00:00<?, ?B/s]

Generating public_test split:   0%|          | 0/10033 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/961 [00:00<?, ? examples/s]

  Загружено примеров: 10033
  Уникальных доменов: 57
Естественные науки: 10033 → 2543 примеров

Примеров по доменам естественных наук:
domain
nutrition                 252
high_school_biology       250
conceptual_physics        167
high_school_chemistry     165
high_school_geography     163
human_aging               152
professional_medicine     148
clinical_knowledge        142
virology                  132
electrical_engineering    119
astronomy                 116
college_biology           112
high_school_physics       111
anatomy                   102
human_sexuality            97
college_chemistry          91
medical_genetics           84
college_physics            75
college_medicine           65

Итого по наукам: 2543 примеров

Стратифицированная выборка: 19 доменов × 26 = 494 примеров + 6 дополнительных
Индексы для исключения сохранены → natural_science_sample_500.ids.json  (500 id)
Выборка сохранена → natural_science_sample_500.json  (500 примеров)

 Примеры по доменам в выбор

In [3]:
import json
import random
from collections import Counter

RANDOM_SEED = 42
INPUT_FILE = "natural_science_sample_500.json"
OUTPUT_FILE = "dpo_natural_science_500.json"

VALID_LETTERS = {"A", "B", "C", "D"}


def build_prompt(row: dict) -> str:
    """Формирует prompt из вопроса и вариантов ответа."""
    return (
        f"{row['text']}\n"
        f"A) {row['option_a']}\n"
        f"B) {row['option_b']}\n"
        f"C) {row['option_c']}\n"
        f"D) {row['option_d']}"
    )


def pick_rejected_balanced(correct_letter: str, rejected_counts: Counter) -> str:
    """
    Выбирает неправильный вариант случайно, но с весами обратно пропорциональными
    текущему счётчику — чтобы выравнивать распределение rejected по ходу.
    """
    wrong = [l for l in ["A", "B", "C", "D"] if l != correct_letter]

    # Веса: чем меньше буква встречалась в rejected, тем выше шанс выбрать её
    max_count = max(rejected_counts[l] for l in wrong) + 1
    weights = [max_count - rejected_counts[l] for l in wrong]

    chosen = random.choices(wrong, weights=weights, k=1)[0]
    rejected_counts[chosen] += 1
    return chosen


def build_dpo_dataset(input_file: str = INPUT_FILE,
                      output_file: str = OUTPUT_FILE,
                      seed: int = RANDOM_SEED):
    random.seed(seed)

    with open(input_file, "r", encoding="utf-8") as f:
        samples = json.load(f)
    print(f"Загружено примеров: {len(samples)}  ←  {input_file}")

    rejected_counts = Counter({"A": 0, "B": 0, "C": 0, "D": 0})
    records = []
    skipped = 0

    for i, row in enumerate(samples):
        correct_letter = str(row["outputs"]).strip().upper()
        if correct_letter not in VALID_LETTERS:
            print(f"  Пропускаем #{i}: неизвестный outputs={row['outputs']}")
            skipped += 1
            continue

        records.append({
            "prompt": build_prompt(row),
            "chosen": correct_letter,
            "rejected": pick_rejected_balanced(correct_letter, rejected_counts),
        })

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print(f"DPO датасет сохранён → {output_file}  ({len(records)} примеров, пропущено: {skipped})")

    chosen_stats = dict(sorted(Counter(r["chosen"] for r in records).items()))
    rejected_stats = dict(sorted(Counter(r["rejected"] for r in records).items()))
    print(f"  chosen   distribution: {chosen_stats}")
    print(f"  rejected distribution: {rejected_stats}")

    return records


if __name__ == "__main__":
    dpo_records = build_dpo_dataset()

    print("\nПример первого DPO примера:")
    print(json.dumps(dpo_records[0], ensure_ascii=False, indent=2))

Загружено примеров: 500  ←  natural_science_sample_500.json
DPO датасет сохранён → dpo_natural_science_500.json  (500 примеров, пропущено: 0)
  chosen   distribution: {'A': 118, 'B': 121, 'C': 122, 'D': 139}
  rejected distribution: {'A': 125, 'B': 127, 'C': 124, 'D': 124}

Пример первого DPO примера:
{
  "prompt": "Если вы верите, что изменения происходят медленно и стабильно на протяжении всей жизни, вы будете придерживаться _____ стороны этого взгляда на развитие.\nA) Механизм\nB) Организм\nC) Непрерывность\nD) Стабильность",
  "chosen": "C",
  "rejected": "B"
}


In [4]:
import json
import random

RANDOM_SEED = 42
INPUT_FILE = "dpo_natural_science_500.json"
OUTPUT_FILE = "dpo_natural_science_500_hard.json"
HARD_NEGATIVE_COUNT = 150


def extract_options(prompt: str) -> dict:
    """Достаёт тексты вариантов ответа из prompt."""
    options = {}
    for line in prompt.split("\n"):
        for letter in ["A", "B", "C", "D"]:
            if line.startswith(f"{letter})"):
                options[letter] = line[3:].strip()
    return options


def add_hard_negatives(input_file: str = INPUT_FILE,
                       output_file: str = OUTPUT_FILE,
                       n: int = HARD_NEGATIVE_COUNT,
                       seed: int = RANDOM_SEED):
    random.seed(seed)

    with open(input_file, "r", encoding="utf-8") as f:
        records = json.load(f)
    print(f"Загружено примеров: {len(records)}  ←  {input_file}")

    # Выбираем n случайных индексов для замены
    hard_indices = set(random.sample(range(len(records)), n))
    print(f"Выбрано под hard negatives: {len(hard_indices)} примеров")

    changed = 0
    failed = 0
    for i, record in enumerate(records):
        if i not in hard_indices:
            continue

        options = extract_options(record["prompt"])
        rejected_letter = record["rejected"]
        rejected_text = options.get(rejected_letter)

        if not rejected_text:
            print(f"  Пропускаем #{i}: не удалось извлечь текст для '{rejected_letter}'")
            failed += 1
            continue

        record["rejected"] = f"{rejected_letter}, {rejected_text}"
        changed += 1

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print(f"Сохранено → {output_file}")
    print(f"  hard negatives: {changed}  |  обычных: {len(records) - changed}  |  ошибок: {failed}")

    # Примеры для проверки
    print("\nПримеры изменённых записей:")
    shown = 0
    for i, record in enumerate(records):
        if i in hard_indices and shown < 2:
            print(f"\n  [{i}] chosen:   {record['chosen']}")
            print(f"       rejected: {record['rejected']}")
            shown += 1

    return records


if __name__ == "__main__":
    records = add_hard_negatives()

Загружено примеров: 500  ←  dpo_natural_science_500.json
Выбрано под hard negatives: 150 примеров
Сохранено → dpo_natural_science_500_hard.json
  hard negatives: 150  |  обычных: 350  |  ошибок: 0

Примеры изменённых записей:

  [3] chosen:   A
       rejected: C, этноцентризма

  [12] chosen:   A
       rejected: B, 47,XY,+13


In [5]:
import json
import random
import pandas as pd
from datasets import load_dataset
from collections import Counter

RANDOM_SEED = 42
EXCLUDED_IDS_FILE = "natural_science_sample_500.ids.json"
EXISTING_DPO_FILE = "dpo_natural_science_500_hard.json"
OUTPUT_FILE = "dpo_2250.json"

SIMPLE_COUNT = 1000
HARD_TEXT_COUNT = 750

VALID_LETTERS = {"A", "B", "C", "D"}
IDX_TO_LETTER = {1: "A", 2: "B", 3: "C", 4: "D"}


# Загрузка и подготовка датасета

def load_rummlu_full():
    print("Загружаем ruMMLU из ai-forever/MERA ...")
    ds = load_dataset("ai-forever/MERA", name="rummlu")
    df = ds["public_test"].to_pandas()
    print(f"  Загружено примеров: {len(df)}")

    inputs_expanded = pd.json_normalize(df["inputs"])
    df = pd.concat([df.drop(columns=["inputs"]), inputs_expanded], axis=1)

    df["domain"] = df["meta"].apply(
        lambda m: m.get("domain", "") if isinstance(m, dict) else ""
    )
    # Достаём id из meta
    df["idx"] = df["meta"].apply(
        lambda m: m.get("id") if isinstance(m, dict) else None
    )
    return df


def exclude_used(df: pd.DataFrame, ids_file: str) -> pd.DataFrame:
    with open(ids_file, "r", encoding="utf-8") as f:
        excluded = set(json.load(f))

    before = len(df)
    df_filtered = df[~df["idx"].isin(excluded)].reset_index(drop=True)
    print(f"Исключено уже использованных: {before} → {len(df_filtered)} примеров")
    return df_filtered


# Построение DPO примеров

def build_prompt(row) -> str:
    return (
        f"{row['text']}\n"
        f"A) {row['option_a']}\n"
        f"B) {row['option_b']}\n"
        f"C) {row['option_c']}\n"
        f"D) {row['option_d']}"
    )


def extract_option_text(row, letter: str) -> str:
    col = f"option_{letter.lower()}"
    return str(row[col]).strip() if col in row and row[col] else ""


def pick_rejected_balanced(correct_letter: str, rejected_counts: Counter) -> str:
    wrong = [l for l in ["A", "B", "C", "D"] if l != correct_letter]
    max_count = max(rejected_counts[l] for l in wrong) + 1
    weights = [max_count - rejected_counts[l] for l in wrong]
    chosen = random.choices(wrong, weights=weights, k=1)[0]
    rejected_counts[chosen] += 1
    return chosen


def build_records(df: pd.DataFrame, n_simple: int, n_hard_text: int, seed: int):
    total_needed = n_simple + n_hard_text
    assert len(df) >= total_needed, (
        f"Недостаточно примеров: доступно {len(df)}, нужно {total_needed}"
    )

    # Стратифицированная выборка по доменам
    domains = df["domain"].unique()
    per_domain = total_needed // len(domains)
    remainder = total_needed % len(domains)

    chunks = []
    for i, domain in enumerate(sorted(domains)):
        domain_df = df[df["domain"] == domain]
        take = per_domain + (1 if i < remainder else 0)
        take = min(take, len(domain_df))
        chunks.append(domain_df.sample(n=take, random_state=seed))

    sampled = pd.concat(chunks).sample(frac=1, random_state=seed).reset_index(drop=True)
    print(f"Стратифицированная выборка: {len(sampled)} примеров из {len(domains)} доменов")

    # Разбиваем индексы на simple и hard_text
    all_indices = list(range(len(sampled)))
    hard_text_indices = set(random.sample(all_indices, n_hard_text))

    rejected_counts = Counter({"A": 0, "B": 0, "C": 0, "D": 0})
    records = []

    for i, (_, row) in enumerate(sampled.iterrows()):
        raw = row["outputs"]
        # outputs может быть буквой или числом
        if isinstance(raw, str) and raw.strip().upper() in VALID_LETTERS:
            correct_letter = raw.strip().upper()
        else:
            correct_letter = IDX_TO_LETTER.get(int(raw))

        if not correct_letter:
            print(f"  Пропускаем #{i}: неизвестный outputs={raw}")
            continue

        rejected_letter = pick_rejected_balanced(correct_letter, rejected_counts)

        if i in hard_text_indices:
            rejected_text = extract_option_text(row, rejected_letter)
            rejected_val = f"{rejected_letter}, {rejected_text}" if rejected_text else rejected_letter
        else:
            rejected_val = rejected_letter

        records.append({
            "prompt": build_prompt(row),
            "chosen": correct_letter,
            "rejected": rejected_val,
        })

    simple_count = sum(1 for r in records if len(r["rejected"]) == 1)
    hard_count = len(records) - simple_count
    print(f"  Простых (буква):         {simple_count}")
    print(f"  Hard text (буква+текст): {hard_count}")

    return records, sampled

# Основной запуск

if __name__ == "__main__":
    random.seed(RANDOM_SEED)

    # Загружаем существующий DPO датасет
    with open(EXISTING_DPO_FILE, "r", encoding="utf-8") as f:
        existing_records = json.load(f)
    print(f"Загружен существующий DPO датасет: {len(existing_records)} примеров  ←  {EXISTING_DPO_FILE}")

    # Загружаем полный ruMMLU и убираем уже использованные
    full_df = load_rummlu_full()
    available_df = exclude_used(full_df, EXCLUDED_IDS_FILE)

    # Строим 1750 новых примеров
    print(f"\nСобираем {SIMPLE_COUNT} простых + {HARD_TEXT_COUNT} hard text примеров ...")
    new_records, sampled_df = build_records(available_df, SIMPLE_COUNT, HARD_TEXT_COUNT, RANDOM_SEED)

    # Сохраняем ids новых примеров для последующего исключения при оценке
    new_ids = sampled_df["idx"].dropna().astype(int).tolist()
    new_ids_file = "dpo_extension_1750.ids.json"
    with open(new_ids_file, "w", encoding="utf-8") as f:
        json.dump(new_ids, f, ensure_ascii=False, indent=2)
    print(f"IDs новых примеров сохранены → {new_ids_file}  ({len(new_ids)} id)")

    # Мёрджим и перемешиваем
    all_records = existing_records + new_records
    random.shuffle(all_records)
    print(f"\nИтого после мёрджа: {len(all_records)} примеров")

    # Статистика
    chosen_stats = dict(sorted(Counter(r["chosen"] for r in all_records).items()))
    simple = sum(1 for r in all_records if len(r["rejected"]) == 1)
    hard_text = sum(1 for r in all_records if len(r["rejected"]) > 1 and "," in r["rejected"])
    print(f"  chosen distribution:  {chosen_stats}")
    print(f"  Простых:              {simple}")
    print(f"  Hard text:            {hard_text}")
    print(f"  Итого:                {len(all_records)}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_records, f, ensure_ascii=False, indent=2)
    print(f"\nСохранено → {OUTPUT_FILE}")

    print("\nПример простого rejected:")
    ex_simple = next(r for r in all_records if len(r["rejected"]) == 1)
    print(json.dumps(ex_simple, ensure_ascii=False, indent=2))

    print("\nПример hard text rejected:")
    ex_hard = next(r for r in all_records if "," in r["rejected"])
    print(json.dumps(ex_hard, ensure_ascii=False, indent=2))

Загружен существующий DPO датасет: 500 примеров  ←  dpo_natural_science_500_hard.json
Загружаем ruMMLU из ai-forever/MERA ...
  Загружено примеров: 10033
Исключено уже использованных: 10033 → 9533 примеров

Собираем 1000 простых + 750 hard text примеров ...
Стратифицированная выборка: 1750 примеров из 57 доменов
  Простых (буква):         1000
  Hard text (буква+текст): 750
IDs новых примеров сохранены → dpo_extension_1750.ids.json  (1750 id)

Итого после мёрджа: 2250 примеров
  chosen distribution:  {'A': 551, 'B': 527, 'C': 554, 'D': 618}
  Простых:              1350
  Hard text:            900
  Итого:                2250

Сохранено → dpo_2250.json

Пример простого rejected:
{
  "prompt": "Твердые материалы, такие как карбид кремния, используемые для шлифовальных кругов, как говорят, являются примерами\nA) ионные кристаллы\nB) сетевые кристаллы\nC) металлические кристаллы\nD) молекулярные кристаллы",
  "chosen": "B",
  "rejected": "A"
}

Пример hard text rejected:
{
  "prompt": "Как